# 示例 07 · 多智能体：路由器（Router）

**来源：** [LangChain 官方文档 — 多智能体路由器](https://docs.langchain.com/oss/python/langchain/multi-agent/router)

---

## 学习目标

完成本 Notebook 后，你将能够：

1. 解释**路由器模式**及其与监督者模式的区别
2. 使用 `Command(goto=...)` 构建**单路由器**，将查询路由到一个专业智能体
3. 使用 `Send` 构建**并行多智能体路由器**，同时将查询扇出到多个智能体
4. 用 `Annotated[list, operator.add]` 归约器收集并行分支的结果
5. 判断什么场景下选择路由器，什么场景下选择监督者

---

## 背景介绍

### 路由器模式

**路由器**是一个预处理步骤：对传入查询分类，然后将其导向合适的智能体。
与监督者不同——监督者每轮都用 LLM 动态决策——路由器只做**一次分类**，然后退出。

```
用户查询
    │
    ▼
┌─────────┐  分类  ┌──────────────────┐
│  路由器  │ ────► │ billing_agent    │  （单路由）
│  节点   │       └──────────────────┘
└─────────┘
    │       扇出  ┌──────────────────┐
    └──────────► │ docs_agent       │
                 │ history_agent    │  （并行路由）
                 │ usecases_agent   │
                 └──────────────────┘
```

### 两种路由模式

| 模式 | API | 适用场景 |
|------|-----|----------|
| **单智能体路由** | `Command(goto="node_name")` | 查询只属于一个领域 |
| **并行扇出** | `Send("node", state)` 列表 | 查询需要多角度同时回答 |

### 路由器 vs. 监督者

| 维度 | 路由器 | 监督者 |
|------|--------|--------|
| 路由决策 | 一次预处理步骤 | 每轮动态 LLM 决策 |
| 并行能力 | 原生支持（Send） | 默认顺序调用 |
| 记忆能力 | 默认无状态 | 有对话记忆 |
| 延迟 | 低（1 次分类调用） | 高（每步 LLM 决策） |
| 适用场景 | 领域清晰、并行调研 | 多步骤复杂任务 |

请**从上到下**依次运行每个单元格。

## 环境配置

In [ ]:
import sys, operator
from pathlib import Path
from typing import TypedDict, Annotated, Literal

# 添加项目根目录，使 common/ 包可被导入
_root = Path().resolve().parent.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, Send
from langchain_core.messages import HumanMessage
from langchain.tools import tool
from langchain.agents import create_agent

from common.llm import create_llm
from common.tracing import create_langfuse_handler, build_langfuse_config, get_langfuse_host

# 初始化 Langfuse 追踪处理器（用于可视化调用链路）
_handler = create_langfuse_handler()

def make_config(session_id: str = 's07', trace_name: str | None = None) -> dict:
    return build_langfuse_config(_handler, session_id=session_id, trace_name=trace_name or session_id)

print("✓ 环境配置完成")

---

## 第一部分 · 单路由器：`Command` 模式

最简单的路由模式：用一次 LLM 调用对查询分类，然后通过
`Command(goto="node_name")` 将执行直接转移到匹配的专业智能体。

```
START → router_node ──► billing_agent   → END
                    ──► technical_agent → END
                    ──► general_agent   → END
```

图中三条路径都已编译，但每次查询**只有一条**被执行。

### 步骤 1a — 定义专业工具和智能体节点

In [ ]:
# ── 各领域模拟工具 ────────────────────────────────────────────────────────────

@tool
def lookup_invoice(invoice_id: str) -> str:
    # 模拟账单数据库：发票 ID → 状态
    db = {
        'INV-001': '已支付 · 299元 · 2024-01-15',
        'INV-002': '已逾期 · 150元 · 到期 2024-02-01',
        'INV-003': '待支付 · 89元  · 到期 2024-03-10',
    }
    return db.get(invoice_id.upper(), f'未找到发票记录：{invoice_id}')

@tool
def check_service_status(service: str) -> str:
    # 模拟服务健康状态表
    statuses = {
        'api':      '✓ 正常运行 — 平均延迟 42ms',
        'database': '✓ 正常运行 — 当前连接 118/500',
        'auth':     '⚠ 降级运行 — 每隔 5 秒重试',
        'cdn':      '✓ 正常运行 — 缓存命中率 94%',
    }
    return statuses.get(service.lower(), f'未知服务：{service}')

print("✓ 工具定义完成")

# ── 专业智能体节点 ────────────────────────────────────────────────────────────

def billing_agent_node(state: dict) -> dict:
    # 账单专家：只配备发票查询工具
    agent = create_agent(
        model=create_llm(),
        tools=[lookup_invoice],
        system_prompt='你是账单支持专员，帮助客户查询发票状态和付款问题，用中文简洁回答。',
    )
    result = agent.invoke(
        {'messages': [{'role': 'user', 'content': state['query']}]},
        config=make_config('s07-billing-cn'),
    )
    return {'answer': f"【账单智能体】{result['messages'][-1].content}"}

def technical_agent_node(state: dict) -> dict:
    # 技术支持：只配备服务状态检查工具
    agent = create_agent(
        model=create_llm(),
        tools=[check_service_status],
        system_prompt='你是技术支持工程师，帮助客户诊断系统问题和检查服务状态，用中文精准回答。',
    )
    result = agent.invoke(
        {'messages': [{'role': 'user', 'content': state['query']}]},
        config=make_config('s07-technical-cn'),
    )
    return {'answer': f"【技术智能体】{result['messages'][-1].content}"}

def general_agent_node(state: dict) -> dict:
    # 通用客服：无需工具，直接用 LLM 知识回答
    agent = create_agent(
        model=create_llm(), tools=[],
        system_prompt='你是热情的客服人员，请用中文简洁专业地回答客户问题。',
    )
    result = agent.invoke(
        {'messages': [{'role': 'user', 'content': state['query']}]},
        config=make_config('s07-general-cn'),
    )
    return {'answer': f"【通用智能体】{result['messages'][-1].content}"}

print("✓ 专业智能体节点定义完成")

### 步骤 1b — 构建路由节点

In [ ]:
# ── 图的状态定义 ──────────────────────────────────────────────────────────────
class SupportState(TypedDict):
    query:  str   # 客户原始查询
    answer: str   # 专业智能体的最终回答

# ── 查询分类辅助函数 ───────────────────────────────────────────────────────────
def classify_query(query: str) -> str:
    # 最小化 prompt，只要求一个单词，节省 token
    llm = create_llm()
    response = llm.invoke([
        HumanMessage(content=(
            '请将以下客服查询归入三类之一：billing（账单）、technical（技术）、general（通用）。\n'
            f'查询：{query}\n'
            '只回复一个英文小写单词。'
        ))
    ])
    cat = response.content.strip().lower().split()[0]
    # 防御性处理：分类结果不合法时默认 general
    return cat if cat in {'billing', 'technical', 'general'} else 'general'

# ── 路由节点 ──────────────────────────────────────────────────────────────────
def router_node(state: SupportState) -> Command[Literal['billing_agent', 'technical_agent', 'general_agent']]:
    # 分类查询，通过 Command(goto=...) 将执行转移到对应智能体节点
    category = classify_query(state['query'])
    print(f'  → 已分类为：[{category}]')
    # Command(goto=...) 告诉 LangGraph 下一步执行哪个节点
    return Command(goto=f'{category}_agent')

print("✓ 路由节点定义完成")

### 步骤 1c — 组装并编译图

In [ ]:
# ── 构建 StateGraph ───────────────────────────────────────────────────────────
builder = StateGraph(SupportState)

# 注册所有节点
builder.add_node('router',           router_node)
builder.add_node('billing_agent',    billing_agent_node)
builder.add_node('technical_agent',  technical_agent_node)
builder.add_node('general_agent',    general_agent_node)

# 图入口：所有查询先经过路由节点
builder.add_edge(START, 'router')

# 各专业智能体完成后直接结束
# 路由节点通过 Command(goto=...) 动态决定路径，无需显式条件边
builder.add_edge('billing_agent',   END)
builder.add_edge('technical_agent', END)
builder.add_edge('general_agent',   END)

# 编译图（无状态路由器不需要 checkpointer）
support_router = builder.compile()
print("✓ 单路由图编译完成")
print('节点列表：', list(support_router.graph.nodes))

### 步骤 1d — 运行路由器

In [ ]:
# 用三类不同查询测试路由器
test_queries = [
    '请帮我查一下发票 INV-002 的状态',
    'auth 服务响应很慢，能帮我检查一下吗？',
    '你们周末有客服支持吗？',
]

for query in test_queries:
    print('=' * 60)
    print(f'查询：{query}')
    result = support_router.invoke(
        {'query': query, 'answer': ''},
        config=make_config('s07-part1-cn'),
    )
    print(f"回答：{result['answer']}")
    print()

---

## 第二部分 · 并行路由：`Send` 扇出模式

有时一个查询需要多个维度同时作答。**扇出模式**使用 `Send`
将查询*同时*分发给多个智能体，再通过 `operator.add` 归约器将所有结果收集到共享状态。

```
                   ┌──► docs_agent    ─┐
START → 扇出路由 ──►│    history_agent ─┼──► (sub_results 自动合并) → END
                   └──► usecases_agent─┘
```

**关键：** `sub_results: Annotated[list[str], operator.add]`
每个并行智能体追加一个字符串；`operator.add` 自动将三个列表合并为一个。

### 步骤 2a — 定义并行状态和智能体节点

In [ ]:
# ── 调研工具（模拟） ──────────────────────────────────────────────────────────

@tool
def search_docs(keyword: str) -> str:
    # 模拟技术文档搜索
    docs = {
        'python':     'Python 是高层次解释型语言，注重可读性和简洁语法。',
        'javascript': 'JavaScript 是事件驱动的单线程 Web 语言。',
        'rust':       'Rust 通过所有权系统在编译期保证内存安全。',
        'go':         'Go 是静态类型语言，为简洁性和并发性而设计。',
    }
    for k, v in docs.items():
        if k in keyword.lower(): return v
    return f'未找到关于 {keyword!r} 的技术文档'

@tool
def search_history(topic: str) -> str:
    # 模拟技术历史数据库
    history = {
        'python':     '由 Guido van Rossum 创建，1991 年首次发布。',
        'javascript': '由 Brendan Eich 在 1995 年用 10 天设计完成。',
        'rust':       '起源于 Mozilla 2006 年的个人项目，2015 年发布 1.0。',
        'go':         '由 Google 的 Pike、Thompson、Griesemer 设计，2009 年发布。',
    }
    for k, v in history.items():
        if k in topic.lower(): return v
    return f'未找到关于 {topic!r} 的历史信息'

@tool
def get_use_cases(technology: str) -> str:
    # 模拟应用场景数据库
    cases = {
        'python':     '数据科学、机器学习、Web 后端（Django/FastAPI）、脚本自动化。',
        'javascript': '前端 UI、Node.js 后端、移动端（React Native）、全栈开发。',
        'rust':       '系统编程、WebAssembly、嵌入式开发、高性能网络服务。',
        'go':         '云基础设施、CLI 工具、微服务、DevOps 工具链。',
    }
    for k, v in cases.items():
        if k in technology.lower(): return v
    return f'未找到 {technology!r} 的应用场景'

print("✓ 调研工具定义完成")

# ── 并行状态（operator.add 归约器） ──────────────────────────────────────────
class ResearchState(TypedDict):
    query: str
    # operator.add 归约器：并行分支各自追加结果，最终合并为完整列表
    sub_results: Annotated[list[str], operator.add]

# ── 并行智能体节点 ─────────────────────────────────────────────────────────────
# 每个节点通过 Send 接收独立状态副本；返回 sub_results 追加到父图状态

def docs_agent_node(state: dict) -> dict:
    # 技术文档智能体：解释该技术「是什么」
    agent = create_agent(
        model=create_llm(), tools=[search_docs],
        system_prompt='你是技术文档专家，请提供准确简洁的技术说明，用中文回答。',
    )
    result = agent.invoke(
        {'messages': [{'role': 'user', 'content': state['query']}]},
        config=make_config('s07-docs-cn'),
    )
    return {'sub_results': [f"【技术说明】{result['messages'][-1].content}"]}

def history_agent_node(state: dict) -> dict:
    # 历史背景智能体：解释该技术「从哪里来」
    agent = create_agent(
        model=create_llm(), tools=[search_history],
        system_prompt='你是技术史专家，请提供技术的历史背景和起源，用中文回答。',
    )
    result = agent.invoke(
        {'messages': [{'role': 'user', 'content': state['query']}]},
        config=make_config('s07-history-cn'),
    )
    return {'sub_results': [f"【历史背景】{result['messages'][-1].content}"]}

def usecases_agent_node(state: dict) -> dict:
    # 应用场景智能体：解释该技术「用在哪里」
    agent = create_agent(
        model=create_llm(), tools=[get_use_cases],
        system_prompt='你是技术顾问，请介绍技术的实际应用场景，用中文回答。',
    )
    result = agent.invoke(
        {'messages': [{'role': 'user', 'content': state['query']}]},
        config=make_config('s07-usecases-cn'),
    )
    return {'sub_results': [f"【应用场景】{result['messages'][-1].content}"]}

print("✓ 并行智能体节点定义完成")

### 步骤 2b — 用 `Send` 构建扇出路由器

In [ ]:
# ── 扇出路由函数（条件边函数，不是节点） ─────────────────────────────────────
def fan_out_router(state: ResearchState) -> list:
    # 返回 Send 列表：每个 Send 为对应并行分支创建独立的状态副本
    print(f'  → 扇出路由：将查询同时分发到 文档 / 历史 / 场景 三个智能体...')
    return [
        Send('docs_agent',     {'query': state['query'], 'sub_results': []}),
        Send('history_agent',  {'query': state['query'], 'sub_results': []}),
        Send('usecases_agent', {'query': state['query'], 'sub_results': []}),
    ]

# ── 构建并行路由图 ─────────────────────────────────────────────────────────────
builder2 = StateGraph(ResearchState)
builder2.add_node('docs_agent',     docs_agent_node)
builder2.add_node('history_agent',  history_agent_node)
builder2.add_node('usecases_agent', usecases_agent_node)

# 从 START 通过 fan_out_router 条件边扇出到三个并行节点
# 第三个参数声明所有可能的目标节点（供 LangGraph 静态图分析）
builder2.add_conditional_edges(
    START, fan_out_router,
    ['docs_agent', 'history_agent', 'usecases_agent'],
)
# 各并行节点完成后直接进入 END；operator.add 已自动合并各分支结果
builder2.add_edge('docs_agent',     END)
builder2.add_edge('history_agent',  END)
builder2.add_edge('usecases_agent', END)

research_router = builder2.compile()
print("✓ 并行扇出路由器编译完成")

### 步骤 2c — 运行并行路由器

In [ ]:
# 三个智能体并行运行，各自将结果追加到 sub_results，operator.add 自动合并
query = '请介绍 Python 编程语言'
print(f'查询：{query}')
print('=' * 60)

result = research_router.invoke(
    {'query': query, 'sub_results': []},
    config=make_config('s07-part2-cn'),
)

print('\n── 各并行智能体的调研结果 ─────────────────────────────')
for entry in result['sub_results']:
    print(f'\n{entry}')
print(f"\n✓ 共收到 {len(result['sub_results'])} 个智能体的响应")

---

## 第三部分 · 路由器 vs. 监督者 — 如何选择

### 决策指南

```
路由逻辑是否事先可知（规则或单次 LLM 分类）？
    是  → 路由器
    否  → 监督者

多个智能体是否需要同时回答？
    是  → 路由器（并行扇出 + Send）
    否  → 路由器（单路由 + Command）或监督者

编排者是否需要记住之前的工具调用？
    是  → 监督者
    否  → 路由器

每次查询需要调用的智能体数量是否不定？
    是  → 监督者
    否  → 路由器
```

### 横向对比

| 维度 | 路由器 | 监督者 |
|------|--------|--------|
| **路由决策** | 预处理（1 次 LLM 或规则） | 每轮动态 LLM 决策 |
| **并行能力** | 原生支持（`Send`） | 默认顺序工具调用 |
| **记忆能力** | 默认无状态 | 有对话记忆 |
| **每次调用数** | 固定：1 个或 N 个 | 变化：运行时决定 |
| **适用场景** | 领域清晰、并行调研 | 多步骤、深度未知 |
| **LangGraph API** | `Command(goto=...)`, `Send(...)` | `create_agent` + 子智能体工具 |
| **延迟** | 低（无每步 LLM 开销） | 高（每步需要 LLM 决策） |

---

## 总结

| 概念 | API | 说明 |
|------|-----|------|
| 图状态 | `TypedDict` | 所有节点共享的类型化状态 |
| 状态归约器 | `Annotated[list, operator.add]` | 自动合并并行分支的输出 |
| 单路由 | `Command(goto="node_name")` | 从路由*节点*返回 |
| 并行路由 | `[Send("node", state), ...]` | 从*条件边*函数返回 |
| 图入口（普通） | `add_edge(START, "router")` | 经过路由节点 |
| 图入口（扇出） | `add_conditional_edges(START, fn, [...])` | 直接从 START 扇出 |

### 核心要点

1. **路由器 = 分类一次，然后退出** — LLM 开销最小（只需一次分类调用）
2. **`Command(goto=...)` 用于单路由** — 从节点返回，非条件边
3. **`Send` 用于并行扇出** — 每个 `Send` 以独立状态副本启动一个分支
4. **`operator.add` 是粘合剂** — 静默合并每个并行分支的输出为一个列表
5. **路由器默认无状态** — 仅在需要多轮记忆时才添加 `InMemorySaver` 检查点

In [ ]:
print(f'追踪链路查看地址：{get_langfuse_host()}')